# Ekstraksi Nilai Resistivitas untuk setiap Kedalaman 

# 1. Import Library & Pengaturan Direktori Utama

In [4]:
import os
import re  
import numpy as np
import pandas as pd

base_amt_dir = r"C:\KULIAH\KULTER 2\MT\trial_datashifa\edis\AMT"
sitelist_path = r"C:\KULIAH\KULTER 2\MT\trial_datashifa\Sitelist.dat"
model_dir = os.path.join(base_amt_dir, "02. PLOT L-CURVE & 1D MODEL")
output_dir = os.path.join(base_amt_dir, "03. EXTRACT DEPTH")

os.makedirs(output_dir, exist_ok=True)

# 2. Ekstraksi Sitelist & Identifikasi File CSV

In [5]:
def parse_sitelist(filepath):
    coords = {}
    if not os.path.exists(filepath): return coords
    with open(filepath, 'r', errors='ignore') as f:
        for line in f:
            parts = line.strip().split()
            if not parts or line.startswith(('!', '#')) or len(parts) < 3: continue
            try:
                coords[parts[0].upper().replace('.EDI', '')] = (
                    float(parts[1].replace(',', '.')),
                    float(parts[2].replace(',', '.')),
                    float(parts[3].replace(',', '.')) if len(parts) >= 4 else "N/A"
                )
            except ValueError: continue
    return coords

site_coords = parse_sitelist(sitelist_path)
csv_files = [f for f in os.listdir(model_dir) if f.endswith('.csv') and 'PARAMETER' not in f] if os.path.exists(model_dir) else []

In [3]:
def parse_edi_coords(edi_folder):
    """
    Baca LAT, LONG, ELEV langsung dari header >HEAD tiap file .edi.
    Format header (contoh): LAT=-7.806250, LONG=111.623667, ELEV=749.171
    """
    coords = {}
    if not os.path.exists(edi_folder): return coords

    for fname in os.listdir(edi_folder):
        if not fname.lower().endswith('.edi'): continue
        site = os.path.splitext(fname)[0].upper()
        site = re.sub(r'_SSQ_AMT$', '', site) 
        with open(os.path.join(edi_folder, fname), 'r', errors='ignore') as f:
            content = f.read()

        lat_m  = re.search(r'LAT\s*=\s*(-?[\d.]+)', content, re.IGNORECASE)
        lon_m  = re.search(r'LONG\s*=\s*(-?[\d.]+)', content, re.IGNORECASE)
        elev_m = re.search(r'ELEV\s*=\s*(-?[\d.]+)', content, re.IGNORECASE)

        if lat_m and lon_m:
            coords[site] = (
                float(lat_m.group(1)),
                float(lon_m.group(1)),
                float(elev_m.group(1)) if elev_m else 0.0
            )
    return coords

# Ganti sumber koordinat: bukan dari Sitelist.dat, tapi dari folder berisi file .edi asli
edi_source_folder = os.path.join(base_amt_dir, "00. DATA EDI")  # sesuaikan kalau folder .edi mentah lokasinya beda
site_coords = parse_edi_coords(edi_source_folder)

csv_files = [f for f in os.listdir(model_dir) if f.endswith('.csv') and 'PARAMETER' not in f] if os.path.exists(model_dir) else []

In [7]:
import os
import re

base_amt_dir = r"C:\KULIAH\KULTER 2\MT\trial_datashifa\edis\AMT"
edi_source_folder = os.path.join(base_amt_dir, "00. DATA EDI")

def parse_edi_coords(edi_folder):
    coords = {}
    if not os.path.exists(edi_folder): return coords
    for fname in os.listdir(edi_folder):
        if not fname.lower().endswith('.edi'): continue
        site = os.path.splitext(fname)[0].upper()
        site = re.sub(r'_SSQ_AMT$', '', site)
        with open(os.path.join(edi_folder, fname), 'r', errors='ignore') as f:
            content = f.read()
        lat_m  = re.search(r'LAT\s*=\s*(-?[\d.]+)', content, re.IGNORECASE)
        lon_m  = re.search(r'LONG\s*=\s*(-?[\d.]+)', content, re.IGNORECASE)
        elev_m = re.search(r'ELEV\s*=\s*(-?[\d.]+)', content, re.IGNORECASE)
        if lat_m and lon_m:
            coords[site] = (
                float(lat_m.group(1)),
                float(lon_m.group(1)),
                float(elev_m.group(1)) if elev_m else 0.0
            )
    return coords

site_coords = parse_edi_coords(edi_source_folder)

print("Stasiun berhasil ke-parse:", sorted(site_coords.keys()))
print("Jumlah:", len(site_coords), "dari 10 harusnya")
print()

edi_path = os.path.join(edi_source_folder, "MT10_SSQ_AMT.edi")
with open(edi_path, 'r', errors='ignore') as f:
    content = f.read()
idx = content.upper().find('>HEAD')
print(content[idx:idx+800])

Stasiun berhasil ke-parse: ['MT1', 'MT10', 'MT2', 'MT3', 'MT4', 'MT5', 'MT6', 'MT7', 'MT8', 'MT9']
Jumlah: 10 dari 10 harusnya

>HEAD
    ACQBY=ITS
    ACQDATE=2026-08-19
    DATAID=MT10
    ELEV=845.186
    FILEBY=ITS
    LAT=-7.828389
    LOC=None
    LONG=111.645889
    FILEDATE=2026/09/08 01:22:52 UTC
    EMPTY=1e+32
    PROGDATE=2026-09-08
    PROGVERS=MTpy
    COORDINATE_SYSTEM=Geomagnetic North
    DECLINATION=None
    DATUM=WGS84
    PROJECT=None
    SURVEY=Near Ponorogo ID SE Asia Standard Time
    UNITS=[mV/km]/[nT]

>INFO
    ac = 0.000671385 [V]
    azimuth = 0
    cal name = 57653_6A7A7863.scal
    comb filter = 50 HZ
    coordinates = -7Â° 49' 42.2, 111Â° 38' 45.2
    dc = -0.0177002 [V]
    declination = 0Â°
    detected sensor type = MTC-155
    duration = 7h 31m 52s
    editing workbench = 
    electrics = 
    ex = 
    ey = 
    fine robust = NONE
    gps (min - max) = 6 - 10
    hx = 
    hy = 



# 3. VERSI 1: Ekstraksi Seluruh Kedalaman Asli

In [8]:
berhasil_v1 = 0

for f in csv_files:
    site = f.replace('.csv', '').upper()
    df = pd.read_csv(os.path.join(model_dir, f))
    if 'Depth' not in df.columns or 'Resistivity' not in df.columns: continue
    
    lat, lon, elev = site_coords.get(site, ("N/A", "N/A", "N/A"))
    df = df.assign(Station=site, Latitude=lat, Longitude=lon, Elevation=elev)
    df['Resistivity'] = df['Resistivity'].round(4)
    
    df[['Station', 'Depth', 'Resistivity', 'Latitude', 'Longitude', 'Elevation']].to_csv(
        os.path.join(output_dir, f"{site}_EXTRACT DEPTH.csv"), index=False
    )
    print(f"✓ {site: <6} | {len(df): <3} lapis kedalaman diekstrak.")
    berhasil_v1 += 1

print(f"\n✅ Selesai: {berhasil_v1} file berhasil diekstrak.")

✓ MT1    | 51  lapis kedalaman diekstrak.
✓ MT10   | 51  lapis kedalaman diekstrak.
✓ MT2    | 51  lapis kedalaman diekstrak.
✓ MT3    | 51  lapis kedalaman diekstrak.
✓ MT4    | 51  lapis kedalaman diekstrak.
✓ MT5    | 51  lapis kedalaman diekstrak.
✓ MT6    | 51  lapis kedalaman diekstrak.
✓ MT7    | 51  lapis kedalaman diekstrak.
✓ MT8    | 51  lapis kedalaman diekstrak.
✓ MT9    | 31  lapis kedalaman diekstrak.

✅ Selesai: 10 file berhasil diekstrak.


# 4. VERSI 2: Ekstraksi Kedalaman dengan Toleransi Dinamis (Interval 250m)

In [9]:
berhasil_v2 = 0

for f in csv_files:
    site = f.replace('.csv', '').upper()
    df = pd.read_csv(os.path.join(model_dir, f))
    if 'Depth' not in df.columns or 'Resistivity' not in df.columns: continue
    
    lat, lon, elev = site_coords.get(site, ("N/A", "N/A", "N/A"))
    max_d = df['Depth'].max()
    targets = np.arange(250, max_d + 250, 250)
    
    hasil = []
    for t in targets:
        baris = df[(df['Depth'] >= t - 125) & (df['Depth'] < t + 125)]
        if not baris.empty:
            m_res = baris['Resistivity'].mean()
            hasil.append([site, float(t), round(m_res, 3), round(np.log10(m_res), 3) if m_res > 0 else None, lat, lon, elev])
            
    if hasil:
        df_out = pd.DataFrame(hasil, columns=["Station", "Depth", "Resistivity Mean", "Log Resistivity", "Latitude", "Longitude", "Elevation"])
        df_out.sort_values("Depth").to_csv(os.path.join(output_dir, f"{site}_EXTRACT with TOLERANCE DEPTH.csv"), index=False)
        print(f"✓ {site: <6} | Max Depth: {max_d: >7.1f} m | {len(df_out): <3} baris diekstrak.")
        berhasil_v2 += 1

print(f"\n✅ Selesai: {berhasil_v2} file berhasil diekstrak.")

✓ MT1    | Max Depth:  7730.0 m | 14  baris diekstrak.
✓ MT10   | Max Depth:  7730.0 m | 14  baris diekstrak.
✓ MT2    | Max Depth:  7730.0 m | 14  baris diekstrak.
✓ MT3    | Max Depth:  7730.0 m | 14  baris diekstrak.
✓ MT4    | Max Depth:  7730.0 m | 14  baris diekstrak.
✓ MT5    | Max Depth:  7730.0 m | 14  baris diekstrak.
✓ MT6    | Max Depth:  7730.0 m | 14  baris diekstrak.
✓ MT7    | Max Depth:  7730.0 m | 14  baris diekstrak.
✓ MT8    | Max Depth:  7730.0 m | 14  baris diekstrak.
✓ MT9    | Max Depth:  7310.0 m | 12  baris diekstrak.

✅ Selesai: 10 file berhasil diekstrak.
